# Datalake

Leer y escribir archivos **pequeños** desde tareas con `ctx.datalake` (pandas). Para transformar
archivos grandes use Synapse: [03_synapse.ipynb](03_synapse.ipynb).

In [ ]:
import logging
import os
import sys
import uuid
from pathlib import Path

from dotenv import load_dotenv

# Raíz del repo (el notebook vive en docs/) y variables locales: .env y .env.synapse (ignorados por git).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT))
for env_file in ('.env', '.env.synapse'):
    load_dotenv(ROOT / env_file, override=True)

# Solo en el notebook: ejecutar tareas que no están en el YAML. En la Function las tareas viven en el YAML.
os.environ['TAREA_DEBE_CONFIGURACION'] = 'false'

from centraal_dataframework.runner import reset_runner  # noqa: E402
from centraal_dataframework.settings import clear_settings_cache  # noqa: E402

clear_settings_cache()  # relee las variables recién cargadas
reset_runner()
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

HAY_STORAGE = bool(os.environ.get('AZURE_STORAGE_CONNECTION_STRING') or os.environ.get('STORAGE_ACCOUNT_NAME'))

# Variables (en .env, ver docs/datalake.env.example):
#   AZURE_STORAGE_CONNECTION_STRING  con AccountKey      -> autenticación por clave
#   STORAGE_ACCOUNT_NAME             sin connection str. -> identidad (az login / Managed Identity)
#   DATALAKE_TEST_PATH               contenedor/carpeta donde se puede escribir (se crea y borra una subcarpeta)
RUTA = os.environ.get('DATALAKE_TEST_PATH', '').strip().strip('/')
if not HAY_STORAGE:
    raise RuntimeError('Configure AZURE_STORAGE_CONNECTION_STRING o STORAGE_ACCOUNT_NAME en .env')
if not RUTA:
    raise RuntimeError('Configure DATALAKE_TEST_PATH=contenedor/carpeta en .env')

BASE = f'{RUTA}/cdf_notebook_{uuid.uuid4().hex[:8]}'  # carpeta temporal de este notebook
print('carpeta de trabajo:', BASE)

## 1. Cliente

In [ ]:
from centraal_dataframework.resources import get_datalake

# Fuera de una tarea (explorar) se usa get_datalake(); dentro de una tarea, ctx.datalake.
# Con AccountKey usa la clave; si no, la identidad (DefaultAzureCredential).
datalake = get_datalake()
print(datalake.datalake_name)

## 2. Escribir y leer

In [ ]:
import pandas as pd

ventas = pd.DataFrame(
    {
        'fecha': ['2026-10-01', '2026-10-01', '2026-10-02'],
        'canal': ['web', 'tienda', 'web'],
        'valor': [100.0, 250.5, 80.0],
    }
)

# Rutas: contenedor/carpeta/archivo.ext. La extensión debe coincidir con el método (.csv, .parquet, ...).
# write_* y read_* envuelven pandas: aceptan los mismos argumentos (sep, index, usecols, columns, dtype, ...).
datalake.write_csv(ventas, f'{BASE}/ventas.csv', sep='|', index=False)
datalake.write_parquet(ventas, f'{BASE}/ventas.parquet', index=False)

print(datalake.read_csv(f'{BASE}/ventas.csv', sep='|'))
print(datalake.read_parquet(f'{BASE}/ventas.parquet', columns=['canal', 'valor']))  # Parquet: solo esas columnas

In [ ]:
# Varios archivos con el mismo esquema: una lista de rutas se lee como un solo DataFrame.
for fecha, grupo in ventas.groupby('fecha'):
    datalake.write_csv(grupo, f'{BASE}/diario/ventas_{fecha}.csv', index=False)

archivos = [f'{BASE}/diario/ventas_{fecha}.csv' for fecha in ventas['fecha'].unique()]
print(datalake.read_csv(archivos))

# Carpetas con particiones Hive (carpeta/columna=valor/archivo.csv): datalake.read_csv_with_partition('carpeta/').
# Solo funciona con AccountKey.

In [ ]:
# Link temporal de solo lectura para compartir un archivo (p.ej. en un correo). Requiere AccountKey.
if datalake.fs.account_key:
    print(datalake.generar_url_con_sas_token(f'{BASE}/ventas.csv', 1, 'hour'))
else:
    print('SALTADO: el link SAS requiere AZURE_STORAGE_CONNECTION_STRING con AccountKey')

## 3. Tareas

In [ ]:
from centraal_dataframework import TaskContext, task


# uses declara lo que la tarea carga en su contexto. El cliente se crea al primer uso de ctx.datalake.
@task(uses=['datalake'])
def resumir_ventas(ctx: TaskContext):
    """Lee un archivo, lo resume y escribe el resultado."""
    p = ctx.params  # del YAML, de la cola o de la llamada
    df = ctx.datalake.read_csv(p['origen'], sep='|', usecols=['canal', 'valor'])  # solo las columnas necesarias
    resumen = df.groupby('canal', as_index=False)['valor'].sum()
    ctx.datalake.write_csv(resumen, p['destino'], index=False)
    ctx.logger.info('origen=%s filas=%s canales=%s', p['origen'], len(df), len(resumen))  # métricas, no filas
    return len(resumen)


# pandas carga el archivo completo en la memoria de la Function (3,5 GB en Premium EP1, compartida por la
# tarea en curso): solo archivos pequeños. Ver docs/CAPACIDAD.md.
# Llamada directa para probar; en la Function la ejecuta el runner.
resumir_ventas(params={'origen': f'{BASE}/ventas.csv', 'destino': f'{BASE}/resumen.csv'})

In [ ]:
from centraal_dataframework import RecursoNoDeclarado


@task(uses=['datalake'])
def usa_synapse_sin_declarar(ctx: TaskContext):
    return ctx.synapse  # no está en uses


# Usar algo no declarado falla aunque esté configurado.
try:
    usa_synapse_sin_declarar()
except RecursoNoDeclarado as exc:
    print(exc)

## 4. Otro lago: conexión con nombre

In [ ]:
from centraal_dataframework.registry import check_uses


# 'datalake:curado' solo usa variables con sufijo __CURADO (AZURE_STORAGE_CONNECTION_STRING__CURADO o
# STORAGE_ACCOUNT_NAME__CURADO). Nunca hereda la cuenta por defecto: no escribe en el lago equivocado.
@task(uses=['datalake', 'datalake:curado'])
def publicar_curado(ctx: TaskContext):
    """Lee del lago por defecto y escribe en el lago curado."""
    df = ctx.datalake.read_csv(ctx.params['origen'], sep='|')
    ctx.resource('datalake', 'curado').write_parquet(df, ctx.params['destino'], index=False)
    return len(df)


# Revisa la configuración sin conectarse y nombra la variable que falta.
print(check_uses(['datalake', 'datalake:curado']) or 'configuración completa')

## 5. Runner

In [ ]:
from centraal_dataframework.runner import get_runner

runner = get_runner()

# Igual que `python -m centraal_dataframework check-resources -m mi_paquete.tareas`: tareas con problemas.
print(runner.check_resources())

# Como en la Function: valida recursos, mezcla params del YAML con los de la llamada y notifica (aquí no).
params = {'origen': f'{BASE}/ventas.csv', 'destino': f'{BASE}/resumen.csv'}
runner.run_task('resumir_ventas', params=params, notify=False)

## Limpieza

In [ ]:
datalake.fs.rm(BASE, recursive=True)
print('borrada:', BASE)